<a href="https://colab.research.google.com/github/luciapastorfontalba-arch/Optimization/blob/main/PhaseII.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def PhaseII(Op, f, A, b , sign_c, sign_f):
  # Op:type of optimization (max,min,optimal)
  # f: coeficients of the function
  # A: matrix of coefficients of constrains
  # b: right-hand side coefficients of constrains
  # sign_c:sign of the constrains (e.g., '<=', '>=', '=')
  # sign_f:sing of the variables (e.g., '>=0')

    print_canonical_problem(Op, f, A, b, sign_c, sign_f)
  #Maximize, optimize o minimize
    if Op== [1,0]:
      sense = 'max'
    elif Op==[0,1]:
      sense = 'min'
      f = -f; #fonvertir a max para simplex
    elif Op== [0,0]:
        sense = 'opt'
    else:
        raise ValueError('Código de operación no válido')



    #Canonnica
    for i in range(0,sign_c.shape[0]): #constrains
      if  not np.array_equal(sign_c[i,:],[0, 1]):#<=

        if np.array_equal(sign_c[i,:],[1, 0]):#>=
              A[i,:]=-A[i,:]
              b[i]=-b[i]
        elif np.array_equal(sign_c[i,:],[0, 0]):#=
              A = np.vstack((A, -A[i, :]))
              b = np.append(b, -b[i])


        else:
            raise ValueError('Codigo de operación no valido')

     #Get the estandar form
    for i in range(sign_f.shape[0]-1,0,-1): #signo variable
      if not np.array_equal(sign_f[i,:], [1, 0]):#x>= 0
          if  np.array_equal(sign_f[i,:],[0, 1]):
            A[:,i]=-A[:,i]
            f[i]=-f[i]
          elif np.array_equal(sign_f[i,:], [0, 0]):# x free

            W=-A[:,i]
            A=np.hstack((A[:, :i+1], W, A[:, i+1:]))
            f=np.insert(f, i + 1, -f[i])




    SI= np.identity(A.shape[0])
    A=np.hstack((A, SI))
    f= np.concatenate((f,np.zeros(A.shape[0])))
    z=0

    #Phase II



    num_var=sign_f.shape[0]


    basis = np.arange(num_var, A.shape[1])

    no_basis= np.arange(0, num_var)

    names_inc = [f"x{i+1}" for i in range(num_var)]
    num_slack = A.shape[1] - num_var
    names_slack = [f"s{i+1}" for i in range(num_slack)]
    var_names = np.array(names_inc + names_slack)

    if  np.any(b<0):

        print(" The problem is unfeasible so we  have to implement PhaseI")


    else:
        print("The problem is feasible")
        y=False
        mul= False # in case we get multiple solution with 2 variables
        zs=0       # to keep trak of the number of z that repeats and avoid looping
        z1= np.inf # to compare the previus z
        bland = False

        iteration=0

        while not y:
            iteration +=1;

            # Chek if we can continue iterating
            if not np.any(f>0) and  not mul:
                print("It is already an optimal solution")
                y= True
                break


            #  Step 1: Greatest positive indicator

            if bland:
                ind = np.where(f > 0)[0]
                col_p= ind[0]

            else:
                ind = np.max(f)
                col_p= np.argmax(f)

            if not np.any(A[:, col_p]>0):
                print("Unbounded solution")
                y=True
                break



            # Step 2: we choose the pivot
            col_pivot = A[:, col_p].astype(float).copy()
            non_row=(col_pivot <= 0)
            col_pivot[non_row] = 1
            div= b/col_pivot
            div[non_row]=np.inf
            min_val = np.min(div)
            ind = np.where(div == min_val)[0]
            b_ind= basis[ind]
            min_ind=ind[np.argmin(b_ind)]
            row_p= min_ind
            pivot= A[row_p, col_p]


            #Step 3: we start pivoting

            b[row_p] = b[row_p]/ pivot
            A[row_p,:]=A[row_p, :] / pivot


            for i in range(0, A.shape[0]):
                if i!=row_p :
                    b[i] = b[i]  - A[i,col_p] * b[row_p]
                    A[i,:] = A[i,:] - A[i,col_p] * A[row_p,:]



            z=z-f[col_p]*b[row_p]
            f= f-f[col_p]*A[row_p,:]


            #Step 4: Change the basic variable
            idx = np.where(no_basis == col_p)[0][0]
            new_var=no_basis[idx]

            no_basis[idx]=basis[row_p]

            basis[row_p]= new_var


            #Step 5: Analyze the results
           # implementar contador de z=zs y bland's rule

            if z==z1:
                zs+= 1
            else:
                zs=0
            z1=z
            if np.max(f)<=0:

                if np.any(f[no_basis]==0):
                    print("We have multiple optimal solutions")
                    if num_var>=3 or mul:
                        y=True
                        break
                    else:
                        print("We need an iteration to get the vertex")
                        sol1_f= f.copy()
                        sol1_b=b.copy()
                        sol1_no_basis=no_basis.copy()
                        sol1_basis=basis.copy()
                        sol1_A=A.copy()
                        sol1_z=z
                        mul=True
                if not np.any(f[no_basis]==0):
                    print("We have reached an optimal solution")
                    y=True
                    break
            # To avoid looping we implement Bland's rule
            if zs>= 6:
                bland = True
                print(" We implement Bland to get out od the looping")






    if mul:
        v1 = np.zeros(sol1_A.shape[1])
        v1[sol1_basis] = sol1_b

        v2 = np.zeros(A.shape[1])
        v2[basis] = b


        print("\n[Extreme Vertex 1]")
        for name, val in zip(var_names[:num_var], v1[:num_var]):
            print(f"  {name} = {val:.4f}")
        print(f"  Z = {sol1_z:.4f}")

        print("\n[Extreme Vertex 2]")
        for name, val in zip(var_names[:num_var], v2[:num_var]):
            print(f"  {name} = {val:.4f}")
        print(f"  Z = {z:.4f}")

        print("\n[Infinitely Many Optimal Solutions Segment]")
        print("Any point along the convex combination segment:")
        # v1[0] y v1[1] son x1 y x2 del primer vértice
        # v2[0] y v2[1] son x1 y x2 del segundo vértice
        print(f"  (x1, x2) = λ*({v1[0]:.2f}, {v1[1]:.2f}) + (1-λ)*({v2[0]:.2f}, {v2[1]:.2f})")
        print("  for all λ ∈ [0, 1]")

    else:

        print("Variables in basis:", var_names[basis])
        print("\n--- Final solution ---")
        for var, val in zip(var_names[basis], b):
            print(f"  {var} = {val:.4f}")

        print(f"  Z = {z:.4f}")






In [ ]:
def print_canonical_problem(Op, f, A, b, sign_c, sign_f):
    # Detectar el tipo de optimización
    opt_type = "Max" if Op[0] == 1 else "Min"

    # 1. Construir la Función Objetivo
    obj_terms = []
    for i, coeff in enumerate(f):
        if coeff == 0:
            continue
        sign = "+" if coeff > 0 else "-"
        abs_coeff = abs(coeff)
        coeff_str = f"{abs_coeff:g}" if abs_coeff != 1 else ""

        if not obj_terms:
            term = f"{coeff:g}*x{i+1}" if abs_coeff != 1 else f"{'-' if coeff < 0 else ''}x{i+1}"
        else:
            term = f"{sign} {coeff_str}x{i+1}" if coeff_str else f"{sign} x{i+1}"
        obj_terms.append(term)

    obj_str = " ".join(obj_terms)

    print("=======================================================")
    print(f"  PROBLEM FORMULATION ({opt_type.upper()})")
    print("=======================================================")
    print(f"  {opt_type} Z = {obj_str}")
    print("  s.t.")

    # Mapeo de signos de restricciones [[1,0]->'<=', [0,1]->'>=', [0,0]->'=']
    rel_map = {(1, 0): "<=", (0, 1): ">=", (0, 0): "="}

    # 2. Construir las Restricciones
    for row_idx, row in enumerate(A):
        row_terms = []
        for col_idx, coeff in enumerate(row):
            if coeff == 0:
                continue
            sign = "+" if coeff > 0 else "-"
            abs_coeff = abs(coeff)
            coeff_str = f"{abs_coeff:g}" if abs_coeff != 1 else ""

            if not row_terms:
                term = f"{coeff:g}*x{col_idx+1}" if abs_coeff != 1 else f"{'-' if coeff < 0 else ''}x{col_idx+1}"
            else:
                term = f"{sign} {coeff_str}x{col_idx+1}" if coeff_str else f"{sign} x{col_idx+1}"
            row_terms.append(term)

        rel_tuple = tuple(sign_c[row_idx])
        rel_str = rel_map.get(rel_tuple, "<=")
        lhs = " ".join(row_terms)
        print(f"    {lhs} {rel_str} {b[row_idx]:g}")

    # 3. No negatividad
    var_names = [f"x{i+1}" for i in range(len(f))]
    print(f"    {', '.join(var_names)} >= 0")
    print("=======================================================\n")

In [ ]:
# Looping Beale's example
Op = [1, 0]  # Maximize
f = np.array([10.0, -57.0, -9.0, -24.0])
A = np.array([
    [0.5, -5.5, -2.5, 9.0],
    [0.5, -1.5, -0.5, 1.0],
    [1.0,  0.0,  0.0, 0.0]
])
b = np.array([0.0, 0.0, 1.0])
sign_c = np.array([[0, 1], [0, 1], [0, 1]])  # <=
sign_f = np.array([[1, 0], [1, 0], [1, 0], [1, 0]])

PhaseII(Op, f, A, b, sign_c, sign_f)



  FORMULACIÓN DEL PROBLEMA (MAX)
  Max Z = 10*x1 - 57x2 - 9x3 - 24x4
  s.a.
    0.5*x1 - 5.5x2 - 2.5x3 + 9x4 >= 0
    0.5*x1 - 1.5x2 - 0.5x3 + x4 >= 0
    x1 >= 1
    x1, x2, x3, x4 >= 0

The problem is feasible
 We implement Bland to get out od the looping
 We implement Bland to get out od the looping
 We implement Bland to get out od the looping
 We implement Bland to get out od the looping
 We implement Bland to get out od the looping
 We implement Bland to get out od the looping
We have reached an optimal solution
Variables in basis: ['s1' 'x1' 'x3']

--- Final solution ---
  s1 = 2.0000
  x1 = 1.0000
  x3 = 1.0000
  Z = -1.0000


In [ ]:
# Multiple 3 variables

Op = [1, 0]  # Maximize
f = np.array([2.0, 2.0, 0.0])
A = np.array([
    [1.0, 1.0, 1.0],
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0]
])
b = np.array([4.0, 2.0, 2.0])
sign_c = np.array([[0, 1], [0, 1], [0, 1]])
sign_f = np.array([[1, 0], [1, 0], [1, 0]])

PhaseII(Op, f, A, b, sign_c, sign_f)

  FORMULACIÓN DEL PROBLEMA (MAX)
  Max Z = 2*x1 + 2x2
  s.a.
    x1 + x2 + x3 >= 4
    x1 >= 2
    x2 >= 2
    x1, x2, x3 >= 0

The problem is feasible
Iteration 1: Entering x1 | Leaving x1
We have multiple optimal solutions
Variables in basis: ['x2' 'x1' 's3']

--- Final solution ---
  x2 = 2.0000
  x1 = 2.0000
  s3 = 0.0000
  Z = -8.0000


In [ ]:
# Degenerated simplex without looping

Op = [1, 0]  # Maximize
f = np.array([3.0, 9.0])
A = np.array([
    [1.0, 4.0],
    [1.0, 2.0]
])
b = np.array([8.0, 4.0])
sign_c = np.array([[0, 1], [0, 1]])
sign_f = np.array([[1, 0], [1, 0]])

PhaseII(Op, f, A, b, sign_c, sign_f)

  FORMULACIÓN DEL PROBLEMA (MAX)
  Max Z = 3*x1 + 9x2
  s.a.
    x1 + 4x2 >= 8
    x1 + 2x2 >= 4
    x1, x2 >= 0

The problem is feasible
We have reached an optimal solution
Variables in basis: ['x2' 'x1']

--- Final solution ---
  x2 = 2.0000
  x1 = 0.0000
  Z = -18.0000


In [ ]:
# simplex

import numpy as np
Op = [1, 0]  # Maximize
f = np.array([3.0, 2.0])
A = np.array([[1.0, 1.0], [2.0, 1.0]])
b = np.array([4.0, 6.0])
sign_c = np.array([[0, 1], [0, 1]])  # [0, 1] es <=
sign_f = np.array([[1, 0], [1, 0]])  # [1, 0] es >= 0

PhaseII(Op, f, A, b, sign_c, sign_f)


The problem is feasible
We have reached an optimal solution
Variables in basis: ['x2' 'x1']

--- Final solution ---
  x2 = 2.0000
  x1 = 2.0000
  Z = -10.0000
Iteration 2: Entering x2 | Leaving x2


In [ ]:
# Multiple solution 2 variables
Op = [1, 0]  # Maximize
f = np.array([2.0, 4.0])
A = np.array([[1.0, 2.0], [1.0, 1.0]])
b = np.array([8.0, 6.0])
sign_c = np.array([[0, 1], [0, 1]])
sign_f = np.array([[1, 0], [1, 0]])

PhaseII(Op, f, A, b, sign_c, sign_f)

  FORMULACIÓN DEL PROBLEMA (MAX)
  Max Z = 2*x1 + 4x2
  s.a.
    x1 + 2x2 >= 8
    x1 + x2 >= 6
    x1, x2 >= 0

The problem is feasible
We have multiple optimal solutions
We need an iteration to get the vertex
We have multiple optimal solutions

[Extreme Vertex 1]
  x1 = 0.0000
  x2 = 4.0000
  Z = -16.0000

[Extreme Vertex 2]
  x1 = 4.0000
  x2 = 2.0000
  Z = -16.0000

[Infinitely Many Optimal Solutions Segment]
Any point along the convex combination segment:
  (x1, x2) = λ*(0.00, 4.00) + (1-λ)*(4.00, 2.00)
  for all λ ∈ [0, 1]
